# Ingest Human Frame Annotations

This notebook reads completed pilot and validation CSVs, checks label validity, derives frame labels, and writes clean human annotation tables for the classifier stage.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "01_classification":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CLASSIFICATION_DIR = PROJECT_ROOT / "data/interim/lsc/classification"
HUMAN_DIR = CLASSIFICATION_DIR / "human_annotation"
OUTPUT_DIR = CLASSIFICATION_DIR / "human_labels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PILOT_COMPLETED = HUMAN_DIR / "frame_pilot_annotation_completed.csv"
VALIDATION_COMPLETED = HUMAN_DIR / "frame_validation_annotation_completed.csv"

EXPECTED_COLUMNS = [
    "annotation_id",
    "context_id",
    "analysis_unit",
    "lsc_year",
    "raw_form",
    "target_sentence_plus_adjacent",
    "clinical_frame_present",
    "lived_experience_frame_present",
    "clinical_evidence",
    "lived_evidence",
    "uncertainty_note",
    "annotation_round",
    "codebook_version",
]

## Load Completed Sheets

Save your annotated CSVs with `_completed.csv` filenames before running this notebook.

In [ ]:
missing = [path for path in [PILOT_COMPLETED, VALIDATION_COMPLETED] if not path.exists()]
if missing:
    print("Completed annotation files not found yet:")
    for path in missing:
        print(f"- {path.relative_to(PROJECT_ROOT)}")
    raise SystemExit("Add completed CSVs, then rerun.")

pilot = pd.read_csv(PILOT_COMPLETED)
validation = pd.read_csv(VALIDATION_COMPLETED)
human = pd.concat([pilot, validation], ignore_index=True)

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(human.columns))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

print(f"Loaded human annotations: {len(human):,}")

## Validate and Derive Frames

The two axes are manually labelled. The four-way frame is derived deterministically.

In [ ]:
def parse_bool(value: object) -> bool | pd.NA:
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower()
    if text in {"true", "t", "yes", "y", "1"}:
        return True
    if text in {"false", "f", "no", "n", "0"}:
        return False
    return pd.NA

for column in ["clinical_frame_present", "lived_experience_frame_present"]:
    human[column] = human[column].map(parse_bool).astype("boolean")

invalid = human.loc[
    human["clinical_frame_present"].isna() | human["lived_experience_frame_present"].isna(),
    ["annotation_id", "clinical_frame_present", "lived_experience_frame_present"],
]
if not invalid.empty:
    raise ValueError(f"Invalid or missing boolean labels: {invalid.head(20).to_dict('records')}")

def derive_frame(row: pd.Series) -> str:
    clinical = bool(row["clinical_frame_present"])
    lived = bool(row["lived_experience_frame_present"])
    if clinical and lived:
        return "mixed"
    if clinical:
        return "clinical_only"
    if lived:
        return "lived_only"
    return "other_non_substantive"

human["derived_frame"] = human.apply(derive_frame, axis=1)

duplicate_ids = human["annotation_id"].duplicated().sum()
duplicate_contexts = human["context_id"].duplicated().sum()
if duplicate_ids or duplicate_contexts:
    raise ValueError(f"Duplicate IDs found: annotation_id={duplicate_ids}, context_id={duplicate_contexts}")

human.groupby(["annotation_round", "analysis_unit", "derived_frame"]).size()

## Save Clean Human Labels

In [ ]:
human_path = OUTPUT_DIR / "frame_human_labels.csv"
pilot_path = OUTPUT_DIR / "frame_human_pilot_labels.csv"
validation_path = OUTPUT_DIR / "frame_human_validation_labels.csv"

human.to_csv(human_path, index=False)
human.loc[human["annotation_round"].eq("pilot")].to_csv(pilot_path, index=False)
human.loc[human["annotation_round"].eq("validation")].to_csv(validation_path, index=False)

print("Wrote clean human labels:")
for path in [human_path, pilot_path, validation_path]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")